In [31]:
def count_tokens(text: str) -> int:
    """Return an approximate token count for the given text."""
    if not text:
        return 0
    return len(re.findall(r"\w+|[^\w\s]", text))


def print_token_count(text: str) -> int:
    """Print and return the token count for the given text."""
    count = count_tokens(text)
    print(f"Token count: {count}")
    return count

In [26]:
import re
import json
import numpy as np
from enum import Enum
from dataclasses import dataclass
from typing import List, Dict
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

class BlockType(Enum):
    PARAGRAPH = "paragraph"
    TABLE = "table"
    LIST = "list"
    HEADING = "heading"
    MIXED = "mixed"

@dataclass
class Block:
    type: BlockType
    text: str

print("Loading embedding model...")
embedder = SentenceTransformer('nomic-ai/nomic-embed-text-v1.5', trust_remote_code=True)

def clean_artifacts(text: str) -> str:
    return text

def parse_blocks(text: str) -> List[Block]:
    table_pattern = re.compile(r'(<table[\s\S]*?</table>)', re.IGNORECASE)
    parts = table_pattern.split(text)
    
    blocks = []
    list_pattern = re.compile(r'^(\s*[-*+]\s+|\s*\d+\.\s+|\s*\([a-zA-Z0-9]{1,4}\)\s+)')
    
    for part in parts:
        part = part.strip()
        if not part: continue
            
        if part.lower().startswith('<table'):
            blocks.append(Block(BlockType.TABLE, part))
        else:
            sub_parts = re.split(r'\n{2,}', part)
            for sp in sub_parts:
                sp = sp.strip()
                if not sp: continue
                
                if re.match(r'^#{1,6}\s+', sp):
                    blocks.append(Block(BlockType.HEADING, sp))
                elif list_pattern.match(sp):
                    blocks.append(Block(BlockType.LIST, sp))
                else:
                    blocks.append(Block(BlockType.PARAGRAPH, sp))
                    
    return blocks

def split_list_by_length(list_text: str, max_chars: int) -> List[str]:
    if len(list_text) <= max_chars:
        return [list_text]
        
    marker_pattern = re.compile(r'(^|\n)(\s*[-*+]\s+|\s*\d+\.\s+|\s*\([a-zA-Z0-9]{1,4}\)\s+)')
    parts = marker_pattern.split(list_text)
    
    items, current_item = [], ""
    for part in parts:
        if marker_pattern.match(part):
            if current_item: items.append(current_item.strip())
            current_item = part
        else:
            current_item += part
    if current_item: items.append(current_item.strip())
        
    chunks, current_chunk, current_len = [], [], 0
    for item in items:
        if current_len + len(item) > max_chars and current_chunk:
            chunks.append("\n".join(current_chunk))
            current_chunk = [item]
            current_len = len(item)
        else:
            current_chunk.append(item)
            current_len += len(item)
            
    if current_chunk:
        chunks.append("\n".join(current_chunk))
    return chunks

def calculate_similarity(text1: str, text2: str) -> float:
    if not text1 or not text2: return 0.0
    emb1 = embedder.encode(text1).reshape(1, -1)
    emb2 = embedder.encode(text2).reshape(1, -1)
    return cosine_similarity(emb1, emb2)[0][0]

def semantic_chunker(text: str, sim_threshold: float = 0.4, 
                     max_chars: int = 1600, min_chars: int = 800) -> List[Dict[str, str]]:
    
    text = clean_artifacts(text)
    
    if len(text) <= max_chars:
        return [{"type": "mixed", "text": text}]
        
    raw_blocks = parse_blocks(text)
    processed_blocks = []
    pending_heading = None
    
    # Pre-processing: Attach headings, group contiguous lists
    for block in raw_blocks:
        if block.type == BlockType.HEADING:
            pending_heading = (pending_heading + "\n\n" + block.text) if pending_heading else block.text
            continue
            
        if pending_heading:
            block.text = pending_heading + "\n\n" + block.text
            pending_heading = None
            
        if processed_blocks and block.type == BlockType.LIST and processed_blocks[-1].type == BlockType.LIST:
            processed_blocks[-1].text += "\n\n" + block.text
        else:
            processed_blocks.append(block)

    final_chunks = []
    current_chunk = []
    current_length = 0
    current_type = None

    def flush_chunk():
        nonlocal current_chunk, current_length, current_type
        if current_chunk:
            final_chunks.append({
                "type": current_type.value if current_type else "mixed",
                "text": "\n\n".join(current_chunk)
            })
            current_chunk = []
            current_length = 0
            current_type = None

    for block in processed_blocks:
        block_len = len(block.text)
        
        # 1. ATOMIC MASSIVE BLOCKS
        # If a block is huge, we must split it ONLY if it's a list. 
        # Tables and paragraphs remain completely unsplit to preserve context/structure.
        if block_len > max_chars:
            flush_chunk()
            if block.type == BlockType.LIST:
                for l in split_list_by_length(block.text, max_chars):
                    final_chunks.append({"type": "list", "text": l})
            else:
                # Appends massive tables as a single unbroken document
                final_chunks.append({"type": block.type.value, "text": block.text})
            continue

        if not current_chunk:
            current_chunk.append(block.text)
            current_length = block_len
            current_type = block.type
        else:
            last_text = current_chunk[-1]
            similarity = 0.0
            if current_type == BlockType.PARAGRAPH and block.type == BlockType.PARAGRAPH:
                similarity = calculate_similarity(last_text, block.text)

            # --- THE DECISION MATRIX ---
            is_under_min = (current_length + block_len) <= min_chars
            
            # Prevent tables from being swallowed by random paragraphs just because they are small
            if (current_type == BlockType.TABLE and block.type != BlockType.TABLE) or \
               (block.type == BlockType.TABLE and current_type != BlockType.TABLE):
                is_under_min = False

            is_semantic_match = (current_type == block.type) and (similarity >= sim_threshold)
            is_tiny_block = block_len < 150 or len(last_text) < 150
            
            # Pack consecutive Tables or consecutive Lists together
            is_structural_packing = (current_type == block.type) and (current_type in (BlockType.TABLE, BlockType.LIST))

            if (is_under_min or is_semantic_match or is_tiny_block or is_structural_packing) and (current_length + block_len) <= max_chars:
                current_chunk.append(block.text)
                current_length += block_len + 2
                if current_type != block.type:
                    current_type = BlockType.MIXED 
            else:
                flush_chunk()
                current_chunk.append(block.text)
                current_length = block_len
                current_type = block.type

    flush_chunk() 
    return final_chunks

Loading embedding model...


<All keys matched successfully>


In [27]:

# # --- DEMO ---
# demo_text = """
# # PART XI
# ## RELATIONS BETWEEN THE UNION AND THE STATES
# ### CHAPTER I.—LEGISLATIVE RELATIONS
# #### Distribution of Legislative Powers

# 245. (1) Subject to the provisions of this Constitution, Parliament may make laws for the whole or any part of the territory of India, and the Legislature of a State may make laws for the whole or any part of the State.

# (2) No law made by Parliament shall be deemed to be invalid on the ground that it would have extra-territorial operation.

# 246. (1) Notwithstanding anything in clauses (2) and (3), Parliament has exclusive power to make laws with respect to any of the matters enumerated in List I in the Seventh Schedule (in this Constitution referred to as the "Union List").

# (2) Notwithstanding anything in clause (3), Parliament, and, subject to clause (1), the Legislature of any State also, have power to make laws with respect to any of the matters enumerated in List II in the Seventh Schedule (in this Constitution referred to as the "State List").

# (3) Subject to clauses (1) and (2), the Legislature of any State has exclusive power to make laws for such State or any part thereof with respect to any of the matters enumerated in List III in the Seventh Schedule (in this Constitution referred to as the "Concurrent List").

# (4) Parliament has power to make laws with respect to any matter for any part of the territory of India not included in a State notwithstanding that such matter is a matter enumerated in the State List.

# <table>
# <tr><th>Entry No.</th><th>Subject Matter</th><th>List Designation</th></tr>
# <tr><td>1</td><td>Defence of India and every part thereof including preparation for defence and all such acts as may be conducive in times of war to its prosecution and after its termination to effective demobilisation.</td><td>Union List</td></tr>
# <tr><td>2</td><td>Naval, military and air forces; any other armed forces of the Union.</td><td>Union List</td></tr>
# <tr><td>3</td><td>Delimitation of cantonment areas, local self-government in such areas, the constitution and powers within such areas of cantonment authorities and the regulation of house accommodation.</td><td>Union List</td></tr>
# <tr><td>4</td><td>Naval, military and air force works.</td><td>Union List</td></tr>
# <tr><td>5</td><td>Arms, firearms, ammunition and explosives.</td><td>Union List</td></tr>
# <tr><td>6</td><td>Atomic energy and mineral resources necessary for its production.</td><td>Union List</td></tr>
# </table>

# To better understand the legislative distribution, the following principles established by the Supreme Court of India apply to constitutional interpretation:

# (a) The principle of territorial nexus requires that there must be a legitimate and real connection between the State and the subject matter of the legislation.

# (b) The doctrine of pith and substance is applied when a law enacted by a legislature is challenged on the ground that it trespasses upon the legislative field of another.

# (c) The doctrine of colourable legislation states that what cannot be done directly, cannot be done indirectly.

# (d) Repugnancy between a Central law and a State law under Article 254 is resolved in favour of the Central law, unless the State law has received the assent of the President.

# (e) Residuary powers of legislation, as specified in Article 248, rest exclusively with Parliament, encompassing any matter not enumerated in the Concurrent List or State List.


# ## (Part XXI.—Temporary, Transitional and Special Provisions)\n\nmembers of that Assembly elected from the tribal areas specified in <sup>1</sup>[Part I] of the table appended to paragraph 20 of the Sixth Schedule and such number of other members of that Assembly as may be specified in the order and for the modifications to be made in the rules of procedure of that Assembly for the constitution and proper functioning of such committee.]\n\n<sup>2</sup>[371C. Special provision with respect to the State of Manipur.—(1) Notwithstanding anything in this Constitution, the President may, by order made with respect to the State of Manipur, provide for the constitution and functions of a committee of the Legislative Assembly of the State consisting of members of that Assembly elected from the Hill Areas of that State, for the modifications to be made in the rules of business of the Government and in the rules of procedure of the Legislative Assembly of the State and for any special responsibility of the Governor in order to secure the proper functioning of such committee.\n\n(2) The Governor shall annually, or whenever so required by the President, make a report to the President regarding the administration of the Hill Areas in the State of Manipur and the executive power of the Union shall extend to the giving of directions to the State as to the administration of the said areas.\n\nExplanation.—In this article, the expression “Hill Areas” means such areas as the President may, by order, declare to be Hill areas.]\n\n<sup>3</sup>[371D. Special provisions with respect to <sup>4</sup>[the State of Andhra Pradesh or the State of Telangana].—<sup>5</sup>[(1) The President may by order made with respect to the State of Andhra Pradesh or the State of Telangana, provide, having regard to the requirement of each State, for equitable opportunities and facilities for the people belonging to different parts of such State, in the matter of public employment and in the matter of education, and different provisions may be made for various parts of the States.]\n\n1. Subs. by the North-Eastern Areas (Reorganisation) Act, 1971 (81 of 1971), s. 71, for \"Part A\" (w.e.f. 21-1-1972).\n\n2. Art 371C ins. by the Constitution (Twenty-seventh Amendment) Act, 1971, s. 5 (w.e.f. 15-2-1972).\n\n3. Art 371D and Art 371E ins. by the Constitution (Thirty-second Amendment) Act, 1973, s. 3 (w.e.f. 1-7-1974).\n\n4. Subs. by the Andhra Pradesh Reorganisation Act, 2014, (6 of 2014), s. 97, for “the State of Andhra Pradesh” (w.e.f. 2-6-2014).\n\n5. Subs. by s. 97, ibid, for cl. (1) (w.e.f. 2-6-2014).



# ## III. LIST OF EXCHANGE OF ENCLAVES BETWEEN INDIA AND BANGLADESH IN PURSUANT TO ARTICLE 1 (12) OF THE AGREEMENT DATED 16TH MAY, 1974 AND THE PROTOCOL TO THE AGREEMENT DATED 6TH SEPTEMBER, 2011

# ## A. EXCHANGEABLE INDIAN ENCLAVES IN BANGLADESH WITH AREA

# <table><tr><td>Sl.</td><td>Name of Chhits No.</td><td>Chhit No.</td><td>Lying within Police station Bangladesh</td><td>Lying within Police station W. Bengal</td><td>Area in acres</td></tr><tr><td>1</td><td>2</td><td>3</td><td>4</td><td>5</td><td>6</td></tr><tr><td></td><td colspan="5">A. Enclaves with independent chhits</td></tr><tr><td>1.</td><td>Garati</td><td>75</td><td>Pochagar</td><td>Haldibari</td><td>58.23</td></tr><tr><td>2.</td><td>Garati</td><td>76</td><td>Pochagar</td><td>Haldibari</td><td>0.79</td></tr><tr><td>3.</td><td>Garati</td><td>77</td><td>Pochagar</td><td>Haldibari</td><td>18</td></tr><tr><td>4.</td><td>Garati</td><td>78</td><td>Pochagar</td><td>Haldibari</td><td>958.66</td></tr><tr><td>5.</td><td>Garati</td><td>79</td><td>Pochagar</td><td>Haldibari</td><td>1.74</td></tr><tr><td>6.</td><td>Garati</td><td>80</td><td>Pochagar</td><td>Haldibari</td><td>73.75</td></tr><tr><td>7.</td><td>Bingimari Part-I</td><td>73</td><td>Pochagar</td><td>Haldibari</td><td>6.07</td></tr></table>

# THE CONSTITUTION OF INDIA  
# (Appendix I)
# <table><tr><td>1</td><td>2</td><td>3</td><td>4</td><td>5</td><td>6</td></tr><tr><td>8.</td><td>Nazirganja</td><td>41</td><td>Boda</td><td>Haldibari</td><td>58.32</td></tr><tr><td>9.</td><td>Nazirganja</td><td>42</td><td>Boda</td><td>Haldibari</td><td>434.29</td></tr><tr><td>10.</td><td>Nazirganja</td><td>44</td><td>Boda</td><td>Haldibari</td><td>53.47</td></tr><tr><td>11.</td><td>Nazirganja</td><td>45</td><td>Boda</td><td>Haldibari</td><td>1.07</td></tr><tr><td>12.</td><td>Nazirganja</td><td>46</td><td>Boda</td><td>Haldibari</td><td>17.95</td></tr><tr><td>13.</td><td>Nazirganja</td><td>47</td><td>Boda</td><td>Haldibari</td><td>3.89</td></tr><tr><td>14.</td><td>Nazirganja</td><td>48</td><td>Boda</td><td>Haldibari</td><td>73.27</td></tr><tr><td>15.</td><td>Nazirganja</td><td>49</td><td>Boda</td><td>Haldibari</td><td>49.05</td></tr><tr><td>16.</td><td>Nazirganja</td><td>50</td><td>Boda</td><td>Haldibari</td><td>5.05</td></tr><tr><td>17.</td><td>Nazirganja</td><td>51</td><td>Boda</td><td>Haldibari</td><td>0.77</td></tr><tr><td>18.</td><td>Nazirganja</td><td>52</td><td>Boda</td><td>Haldibari</td><td>1.04</td></tr><tr><td>19.</td><td>Nazirganja</td><td>53</td><td>Boda</td><td>Haldibari</td><td>1.02</td></tr><tr><td>20.</td><td>Nazirganja</td><td>54</td><td>Boda</td><td>Haldibari</td><td>3.87</td></tr><tr><td>21.</td><td>Nazirganja</td><td>55</td><td>Boda</td><td>Haldibari</td><td>12.18</td></tr><tr><td>22.</td><td>Nazirganja</td><td>56</td><td>Boda</td><td>Haldibari</td><td>54.04</td></tr><tr><td>23.</td><td>Nazirganja</td><td>57</td><td>Boda</td><td>Haldibari</td><td>8.27</td></tr><tr><td>24.</td><td>Nazirganja</td><td>58</td><td>Boda</td><td>Haldibari</td><td>14.22</td></tr><tr><td>25.</td><td>Nazirganja</td><td>60</td><td>Boda</td><td>Haldibari</td><td>0.52</td></tr><tr><td>26.</td><td>Putimari</td><td>59</td><td>Boda</td><td>Haldibari</td><td>122.8</td></tr><tr><td>27.</td><td>Daikhata Chhat</td><td>38</td><td>Boda</td><td>Haldibari</td><td>499.21</td></tr><tr><td>28.</td><td>Salbari</td><td>37</td><td>Boda</td><td>Haldibari</td><td>1188.93</td></tr><tr><td>29.</td><td>Kajal Dighi</td><td>36</td><td>Boda</td><td>Haldibari</td><td>771.44</td></tr><tr><td>30.</td><td>Nataktoka</td><td>32</td><td>Boda</td><td>Haldibari</td><td>162.26</td></tr><tr><td>31.</td><td>Nataktoka</td><td>33</td><td>Boda</td><td>Haldibari</td><td>0.26</td></tr><tr><td>32.</td><td>Beuladanga Chhat</td><td>35</td><td>Boda</td><td>Haldibari</td><td>0.83</td></tr><tr><td>33.</td><td>Balapara Iagrabar</td><td>3</td><td>Debiganj</td><td>Haldibari</td><td>1752.44</td></tr><tr><td>34.</td><td>Bara Khankikharija Citaldaha</td><td>30</td><td>Dimla</td><td>Haldibari</td><td>7.71</td></tr></table>

# (Appendix I)
# <table><tr><td>1</td><td>2</td><td>3</td><td>4</td><td>5</td><td>6</td></tr><tr><td>35.</td><td>Bara Khankikharija Citaldaha</td><td>29</td><td>Dimla</td><td>Haldibari</td><td>36.83</td></tr><tr><td>36.</td><td>Barakhangir</td><td>28</td><td>Dimla</td><td>Haldibari</td><td>30.53</td></tr><tr><td>37.</td><td>Nagarjikobari</td><td>31</td><td>Dimla</td><td>Haldibari</td><td>33.41</td></tr><tr><td>38.</td><td>Kuchlibari</td><td>26</td><td>Patgram</td><td>Mekliganj</td><td>5.78</td></tr><tr><td>39.</td><td>Kuchlibari</td><td>27</td><td>Patgram</td><td>Mekliganj</td><td>2.04</td></tr><tr><td>40.</td><td>Bara Kuchlibari</td><td>Fragment ofJ.L.107 of P.S Mekliganj</td><td>Patgram</td><td>Mekliganj</td><td>4.35</td></tr><tr><td>41.</td><td>Jamaldaha- Balapukhari</td><td>6</td><td>Patgram</td><td>Mekliganj</td><td>5.24</td></tr><tr><td>42.</td><td>Uponchowki kuchlibari</td><td>115/2</td><td>Patgram</td><td>Mekliganj</td><td>0.32</td></tr><tr><td>43.</td><td>Uponchowki kuchlibari</td><td>7</td><td>Patgram</td><td>Mekliganj</td><td>44.04</td></tr><tr><td>44.</td><td>Bhothnri</td><td>11</td><td>Patgram</td><td>Mekliganj</td><td>36.83</td></tr><tr><td>45.</td><td>Balapukhari</td><td>5</td><td>Patgram</td><td>Mekliganj</td><td>55.91</td></tr><tr><td>46.</td><td>Bara Khangir</td><td>4</td><td>Patgram</td><td>Mekliganj</td><td>50.51</td></tr><tr><td>47.</td><td>Bara Khangir</td><td>9</td><td>Patgram</td><td>Mekliganj</td><td>87.42</td></tr><tr><td>48.</td><td>Chhat Bogdokra</td><td>10</td><td>Patgram</td><td>Mekliganj</td><td>41.7</td></tr><tr><td>49.</td><td>Ratanpur</td><td>11</td><td>Patgram</td><td>Mekliganj</td><td>58.91</td></tr><tr><td>50.</td><td>Bogdokra</td><td>12</td><td>Patgram</td><td>Mekliganj</td><td>25.49</td></tr><tr><td>51.</td><td>Fulker Dabri</td><td>Fragment</td><td>Patgram</td><td>Mekliganj</td><td>0.88</td></tr><tr><td></td><td></td><td>of J.L. 107 ofP.S Mekliganj</td><td></td><td></td><td></td></tr></table>

# THE CONSTITUTION OF INDIA  
# (Appendix I)
# <table><tr><td>1</td><td>2</td><td>3</td><td>4</td><td>5</td><td>6</td></tr><tr><td>52.</td><td>Kharkharia</td><td>15</td><td>Patgram</td><td>Mekliganj</td><td>60.74</td></tr><tr><td>53.</td><td>Kharkharia</td><td>13</td><td>Patgram</td><td>Mekliganj</td><td>51.62</td></tr><tr><td>54.</td><td>Lotamari</td><td>14</td><td>Patgram</td><td>Mekliganj</td><td>110.92</td></tr><tr><td>55.</td><td>Bhotbari</td><td>16</td><td>Patgram</td><td>Mekliganj</td><td>205.46</td></tr><tr><td>56.</td><td>Komat Changrabandha</td><td>16A</td><td>Patgram</td><td>Mekliganj</td><td>42.8</td></tr><tr><td>57.</td><td>Komat Changrabandha</td><td>17A</td><td>Patgram</td><td>Mekliganj</td><td>16.01</td></tr><tr><td>58.</td><td>Panisala</td><td>17</td><td>Patgram</td><td>Mekliganj</td><td>137.66</td></tr><tr><td>59.</td><td>Dwarikamari Khasbash</td><td>18</td><td>Patgram</td><td>Mekliganj</td><td>36.5</td></tr><tr><td>60.</td><td>Panisala</td><td>153/P</td><td>Patgram</td><td>Mekliganj</td><td>0.27</td></tr><tr><td>61.</td><td>Panisala</td><td>153/0</td><td>Patgram</td><td>Mekliganj</td><td>18.01</td></tr><tr><td>62.</td><td>Panisala</td><td>19</td><td>Patgram</td><td>Mekliganj</td><td>64.63</td></tr><tr><td>63.</td><td>Panisala</td><td>21</td><td>Patgram</td><td>Mekliganj</td><td>51.4</td></tr><tr><td>64.</td><td>Lotamari</td><td>20</td><td>Patgram</td><td>Mekliganj</td><td>283.53</td></tr><tr><td>65.</td><td>Lotamari</td><td>22</td><td>Patgram</td><td>Mekliganj</td><td>98.85</td></tr><tr><td>66.</td><td>Dwarikamari</td><td>23</td><td>Patgram</td><td>Mekliganj</td><td>39.52</td></tr><tr><td>67.</td><td>Dwarikamari</td><td>25</td><td>Patgram</td><td>Mekliganj</td><td>45.73</td></tr><tr><td>68.</td><td>Chhat Bhothat</td><td>24</td><td>Patgram</td><td>Mekliganj</td><td>56.11</td></tr><tr><td>69.</td><td>Baakata</td><td>131</td><td>Patgram</td><td>Hathabhanga</td><td>2235</td></tr><tr><td>70.</td><td>Baakata</td><td>132</td><td>Patgram</td><td>Hathabhanga</td><td>11.96</td></tr><tr><td>71.</td><td>Baakata</td><td>130</td><td>Patgram</td><td>Hathibhanga</td><td>20.48</td></tr><tr><td>72.</td><td>Bhogramguri</td><td>133</td><td>Patgram</td><td>Hathibhanga</td><td>1.44</td></tr><tr><td>73.</td><td>Chenakata</td><td>134</td><td>Patgram</td><td>Mekliganj</td><td>7.81</td></tr><tr><td>74.</td><td>Banskata</td><td>119</td><td>Patgram</td><td>Mathabanga</td><td>413.81</td></tr><tr><td>75.</td><td>Banskata</td><td>120</td><td>Patgram</td><td>Mathabanga</td><td>30.75</td></tr><tr><td>76.</td><td>Banskata</td><td>121</td><td>Patgram</td><td>Mathabanga</td><td>12.15</td></tr><tr><td>77.</td><td>Banskata</td><td>113</td><td>Patgram</td><td>Mathabanga</td><td>57.86</td></tr><tr><td>78.</td><td>Banskata</td><td>112</td><td>Patgram</td><td>Mathabanga</td><td>315.04</td></tr><tr><td>79.</td><td>Banskata</td><td>114</td><td>Patgram</td><td>Mathabanga</td><td>0.77</td></tr></table>

# THE CONSTITUTION OF INDIA  
# (Appendix I)
# <table><tr><td>1</td><td>2</td><td>3</td><td>4</td><td>5</td><td>6</td></tr><tr><td>80.</td><td>Banskata</td><td>115</td><td>Patgram</td><td>Mathabanga</td><td>29.2</td></tr><tr><td>81.</td><td>Banskata</td><td>122</td><td>Patgram</td><td>Mathabanga</td><td>33.22</td></tr><tr><td>82.</td><td>Banskata</td><td>127</td><td>Patgram</td><td>Mathabanga</td><td>12.72</td></tr><tr><td>83.</td><td>Banskata</td><td>128</td><td>Patgram</td><td>Mathabanga</td><td>2.33</td></tr><tr><td>84.</td><td>Banskata</td><td>117</td><td>Patgram</td><td>Mathabanga</td><td>2.55</td></tr><tr><td>85.</td><td>Banskata</td><td>118</td><td>Patgram</td><td>Mathabanga</td><td>30.98</td></tr><tr><td>86.</td><td>Banskata</td><td>125</td><td>Patgram</td><td>Mathabanga</td><td>0.64</td></tr><tr><td>87.</td><td>Banskata</td><td>126</td><td>Patgram</td><td>Mathabanga</td><td>1.39</td></tr><tr><td>88.</td><td>Banskata</td><td>129</td><td>Patgram</td><td>Mathabanga</td><td>1.37</td></tr><tr><td>89.</td><td>Banskata</td><td>116</td><td>Patgram</td><td>Mathabanga</td><td>16.96</td></tr><tr><td>90.</td><td>Banskata</td><td>123</td><td>Patgram</td><td>Mathabanga</td><td>24.37</td></tr><tr><td>91.</td><td>Banskata</td><td>124</td><td>Patgram</td><td>Mathabanga</td><td>0.28</td></tr><tr><td>92.</td><td>Gotamari Chhit</td><td>135</td><td>Hatibandha</td><td>Sitalkuchi</td><td>126.59</td></tr><tr><td>93.</td><td>Gotamari Chhit</td><td>136</td><td>Hatibandha</td><td>Sitalkuchi</td><td>20.02</td></tr><tr><td>94.</td><td>Banapachai</td><td>151</td><td>Lalmonirhat</td><td>Dinhata</td><td>217.29</td></tr><tr><td>95.</td><td>Banapachai Bhitarkuthi</td><td>152</td><td>Lalmonirhat</td><td>Dinhata</td><td>81.71</td></tr><tr><td>96.</td><td>Dasiar Chhara</td><td>150</td><td>Fulbari</td><td>Dinhata</td><td>1643.44</td></tr><tr><td>97.</td><td>Dakurhat- Dakinirkuthi</td><td>156</td><td>Kurigram</td><td>Dinhata</td><td>14.27</td></tr><tr><td>98.</td><td>Kalamati</td><td>141</td><td>Bhurungamari</td><td>Dinhata</td><td>21.21</td></tr><tr><td>99.</td><td>Bhahobganj</td><td>153</td><td>Bhurungamari</td><td>Dinhata</td><td>31.58</td></tr><tr><td>100.</td><td>Baotikursa</td><td>142</td><td>Bhurungamari</td><td>Dinhata</td><td>45.63</td></tr><tr><td>101.</td><td>Bara Coachulka</td><td>143</td><td>Bhurungamari</td><td>Dinhata</td><td>39.99</td></tr><tr><td>102.</td><td>Gaochulka II</td><td>147</td><td>Bhurungamari</td><td>Dinhata</td><td>0.9</td></tr><tr><td>103.</td><td>Gaochulka I</td><td>146</td><td>Bhurungamari</td><td>Dinhata</td><td>8.92</td></tr><tr><td>104.</td><td>Dighaltari II</td><td>145</td><td>Bhurungamari</td><td>Dinhata</td><td>8.81</td></tr><tr><td>105.</td><td>Dighaltari I</td><td>144</td><td>Bhurungamari</td><td>Dinhata</td><td>12.31</td></tr><tr><td>106.</td><td>Chhoto</td><td>149</td><td>Bhurungamari</td><td>Dinhata</td><td>17.85</td></tr><tr><td></td><td>Garaljhora II</td><td></td><td></td><td></td><td></td></tr></table>

# THE CONSTITUTION OF INDIA  
# (Appendix I)
# <table><tr><td>1</td><td>2</td><td>3</td><td>4</td><td>5</td><td>6</td></tr><tr><td>107.</td><td>Chhoto Garaljhora I 1 chhit* without</td><td>148</td><td>Bhurungamari Patgram</td><td>Dinhata</td><td>35.74</td></tr><tr><td>108.</td><td>name &amp; JL No. at the southern and of JL No. 38 &amp; southern and of JL No. 39 (locally known</td><td></td><td></td><td>Mathabhanga</td><td>3.5</td></tr><tr><td></td><td>as Ashokabari**)</td><td></td><td></td><td></td><td></td></tr><tr><td></td><td></td><td></td><td>Enclaves with Fragmented Chhits</td><td></td><td></td></tr><tr><td>109.</td><td>(i) Bewladanga</td><td>34</td><td>Haldibari</td><td>Boda</td><td>862.46</td></tr><tr><td>110.</td><td>(ii) Bewladanga (i) Kotbhajni</td><td>Fragment</td><td>Haldibari</td><td>Debiganj</td><td>2012.27</td></tr><tr><td></td><td>(ii) Kotbhajni</td><td>2 Fragment</td><td>Haldibari Haldibari</td><td>Debiganj</td><td></td></tr><tr><td></td><td>(iii) Kotbhajni</td><td>Fragment</td><td>Haldibari</td><td>Debiganj Debiganj</td><td></td></tr><tr><td></td><td>(iv) Kotbhajni</td><td>Fragment</td><td>Haldibari</td><td>Debiganj</td><td></td></tr><tr><td>111.</td><td>(i) Dahala</td><td>Khagrabri</td><td>Haldibari</td><td>Debiganj</td><td>2650.35</td></tr><tr><td></td><td>(ii) Dahala</td><td>Fragment</td><td>Haldibari</td><td></td><td></td></tr><tr><td></td><td>(iii) Dahala</td><td></td><td></td><td>Debiganj</td><td></td></tr><tr><td></td><td></td><td>Fragment</td><td>Haldibari</td><td>Debiganj</td><td></td></tr><tr><td></td><td>(iv) Dahala</td><td>Fragment</td><td>Haldibari</td><td>Debiganj</td><td></td></tr></table>

# (Appendix I)
# <table><tr><td></td><td>2</td><td>3</td><td>4</td><td>5</td><td>6</td></tr><tr><td></td><td>(v) Dahala</td><td>Fragment</td><td>Haldibari</td><td>Debiganj</td><td></td></tr><tr><td></td><td>(vi) Dahala</td><td>Fragment</td><td>Haldibari</td><td>Debiganj</td><td></td></tr><tr><td></td><td></td><td></td><td></td><td></td><td>17160.63</td></tr></table>

# The above given details of enclaves have been jointly compared and reconciled with records held by India and Bangladesh during the Indo-Bangladesh Conference held at Calcutta during 9th—12th October, 1996 as well as during joint field inspection at Jalpaiguri (West Bengal) Panchagarh (Bangladesh) sector during 21—24 November, 1996.

# Note: Name of enclave in Sl. No. 108 above has been identified as Ashokabari by joint ground verification during field season 1996-97.

# <table><tr><td>Brig. J.R. Peter Director Land Records &amp; Survey</td><td>Md. Shafi Uddin Director-General, Land Records</td></tr><tr><td>(Ex-Officio) West Bengal, India &amp;</td><td>and Surveys, Bangladesh.</td></tr><tr><td>Director, Eastern Circle Survey of India, Calcutta.</td><td></td></tr></table>

# B. EXCHANGEABLE BANGLADESH ENCLAVES IN INDIA WITH AREA
# <table><tr><td>Sl. No.</td><td>Name of Chhits</td><td>Lying within Police station W. Bengal</td><td>Lying within Police station Bangladesh</td><td>J.L. No.</td><td>Area in acres</td></tr><tr><td>1</td><td>2</td><td>3</td><td>4</td><td>5</td><td>6</td></tr><tr><td></td><td colspan="3">A. Enclaves with independent chhits</td><td></td><td></td></tr><tr><td>1.</td><td>Chhit Kuchlibari</td><td>Mekliganj</td><td>Patgram</td><td>22</td><td>370.64</td></tr><tr><td>2.</td><td>Chhit Land of Kuchlibari</td><td>Mekliganj</td><td>Patgram</td><td>24</td><td>1.83</td></tr><tr><td>3.</td><td>Balapukhari</td><td>Mekliganj</td><td>Patgram</td><td>21</td><td>331.64</td></tr><tr><td>4.</td><td>Chhit Land of Panbari No.2</td><td>Mekliganj</td><td>Patgram</td><td>20</td><td>1.13</td></tr></table>

# (Appendix I)
# <table><tr><td>1</td><td>2</td><td>3</td><td>4</td><td>5</td><td>6</td></tr><tr><td>5.</td><td>Chhit Panbari</td><td>Mekliganj</td><td>Patgram</td><td>18</td><td>108.59</td></tr><tr><td>6.</td><td>Dhabalsati Mirgipur</td><td>Mekliganj</td><td>Patgram</td><td>15</td><td>173.88</td></tr><tr><td>7.</td><td>Bamandal</td><td>Mekliganj</td><td>Patgram</td><td>11</td><td>2.24</td></tr><tr><td>8.</td><td>Chhit Dhabalsati</td><td>Mekliganj</td><td>Patgram</td><td>14</td><td>66.58</td></tr><tr><td>9.</td><td>Dhabalsati</td><td>Mekliganj</td><td>Patgram</td><td>13</td><td>60.45</td></tr><tr><td>10.</td><td>Srirampur</td><td>Mekliganj</td><td>Patgram</td><td>8</td><td>1.05</td></tr><tr><td>11.</td><td>Jote Nijjama</td><td>Mekliganj</td><td>Patgram</td><td>3</td><td>87.54</td></tr><tr><td>12.</td><td>Chhit Land of Jagatber No. 3</td><td>Mathabhanga</td><td>Patgram</td><td>37</td><td>69.84</td></tr><tr><td>13.</td><td>Chhit Land of Jagatber No.1</td><td>Mathabhanga</td><td>Patgram</td><td>35</td><td>30.66</td></tr><tr><td>14.</td><td>Chhit Land of Jagatber No. 2</td><td>Mathabhanga</td><td>Patgram</td><td>36</td><td>27.09</td></tr><tr><td>15.</td><td>Chhit Kokoabari</td><td>Mathabhanga</td><td>Patgram</td><td>47</td><td>29.49</td></tr><tr><td>16.</td><td>Chhit Bhandardaha</td><td>Mathabhanga</td><td>Patgram</td><td>67</td><td>39.96</td></tr><tr><td>17.</td><td>Dhabalguri</td><td>Mathabhanga</td><td>Patgram</td><td>52</td><td>12.5</td></tr><tr><td>18.</td><td>Chhit Dhabalguri</td><td>Mathabhanga</td><td>Patgram</td><td>53</td><td>22.31</td></tr><tr><td>19.</td><td>Chhit Land of Dhabalguri No.3</td><td>Mathabhanga</td><td>Patgram</td><td>70</td><td>1.33</td></tr><tr><td>20.</td><td>Chhit Land of Dhabalguri No. 4</td><td>Mathabhanga</td><td>Patgram</td><td>71</td><td>4.55</td></tr><tr><td>21.</td><td>Chhit Land of Dhabalguri No.5</td><td>Mathabhanga</td><td>Patgram</td><td>72</td><td>4.12</td></tr><tr><td>22.</td><td>Chhit Land of Dhabalguri No. 1</td><td>Mathabhanga</td><td>Patgram</td><td>68</td><td>26.83</td></tr><tr><td>23.</td><td>Chhit Land of Dhabalguri No. 2</td><td>Mathabhanga</td><td>Patgram</td><td>69</td><td>13.95</td></tr><tr><td>24.</td><td>Mahishmari</td><td>Sitalkuchi</td><td>Patgram</td><td>54</td><td>122.77</td></tr><tr><td>25.</td><td>Bura Saradubi</td><td>Sitalkuchi</td><td>Hatibandha</td><td>13</td><td>34.96</td></tr></table>

# (Appendix I)
# <table><tr><td>1</td><td>2</td><td>3</td><td>4</td><td>5</td><td>6</td></tr><tr><td>26.</td><td>Falnapur</td><td>Sitalkuchi</td><td>Patgram</td><td>64</td><td>505.56</td></tr><tr><td>27.</td><td>Amjhol</td><td>Sitalkuchi</td><td>Hatibandha</td><td>57</td><td>1.25</td></tr><tr><td>28.</td><td>Kismat Batrigachh</td><td>Dinhata</td><td>Kaliganj</td><td>82</td><td>209.95</td></tr><tr><td>29.</td><td>Durgapur</td><td>Dinhata</td><td>Kaliganj</td><td>83</td><td>20.96</td></tr><tr><td>30.</td><td>Bansua Khamar Gitaldaha</td><td>Dinhata</td><td>Lalmonirhat</td><td>1</td><td>24.54</td></tr><tr><td>31.</td><td>Poaturkuthi</td><td>Dinhata</td><td>Lalmonirhat</td><td>37</td><td>589.94</td></tr><tr><td>32.</td><td>Paschim Bakalir Chhara</td><td>Dinhata</td><td>Bhurungamari</td><td>38</td><td>151.98</td></tr><tr><td>33.</td><td>Madhya Bakalir Chhara</td><td>Dinhata</td><td>Bhurungamari</td><td>39</td><td>32.72</td></tr><tr><td>34.</td><td>Purba Bakalir Chhara</td><td>Dinhata</td><td>Bhurungamari</td><td>40</td><td>12.23</td></tr><tr><td>35.</td><td>Madhya Masaldanga</td><td>Dinhata</td><td>Bhurungamari</td><td>3</td><td>136.66</td></tr><tr><td>36.</td><td>Madhya Chhit Masaldanga</td><td>Dinhata</td><td>Bhurungamari</td><td>8</td><td>11.87</td></tr><tr><td>37.</td><td>Paschim Chhit Masaldanga</td><td>Dinhata</td><td>Bhurungamari</td><td>7</td><td>7.6</td></tr><tr><td>38.</td><td>Uttar Masaldanga</td><td>Dinhata</td><td>Bhurungamari</td><td>2</td><td>27.29</td></tr><tr><td>39.</td><td>Kachua</td><td>Dinhata</td><td>Bhurungamari</td><td>5</td><td>119.74</td></tr><tr><td>40.</td><td>Uttar Bansjani</td><td>Tufanganj</td><td>Bhurungamari</td><td>1</td><td>47.17</td></tr><tr><td>41.</td><td>Chhat Tilai</td><td>Tufanganj</td><td>Bhurungamari</td><td>17</td><td>81.56</td></tr><tr><td colspan="6">B. Enclaves with Fragmented Chhits</td></tr><tr><td rowspan="4">42.</td><td>(i) Nalgram</td><td>Sitalkuchi</td><td>Patgarm</td><td>65</td><td>1397.34</td></tr><tr><td>(ii) Nalgram (Fragment)</td><td>Sitalkuchi</td><td>Patgarm</td><td>65</td><td></td></tr><tr><td>(iii) Nalgram</td><td>Sitalkuchi</td><td>Patgarm</td><td>65</td><td></td></tr><tr><td>(Fragment)</td><td></td><td></td><td></td><td></td></tr></table>

# THE CONSTITUTION OF INDIA (Appendix I)
# <table><tr><td>1</td><td>2</td><td>3</td><td>4</td><td>5</td><td>6</td></tr><tr><td>43.</td><td>(i) Chhit Nalgram</td><td>Sitalkuchi</td><td>Patgarm</td><td>66</td><td>49.5</td></tr><tr><td rowspan="3">44.</td><td>(ii) Chhit Nalgram (Fragment)</td><td>Sitalkuchi</td><td>Patgarm</td><td>66</td><td></td></tr><tr><td>(i) Batrigachh</td><td>Dinhata</td><td>Kaliganj</td><td>81</td><td>577.37</td></tr><tr><td>(ii) Batrigachh (Fragment)</td><td>Dinhata</td><td>Kaliganj</td><td>81</td><td></td></tr><tr><td rowspan="3">45.</td><td>(iii) Batrigachh (Fragment)</td><td>Dinhata</td><td>Phulbari</td><td>9</td><td></td></tr><tr><td>(i) Karala</td><td>Dinhata</td><td>Phulbari</td><td>9</td><td>269.91</td></tr><tr><td>(ii) Karala (fragment)</td><td>Dinhata</td><td>Phulbari</td><td>9</td><td></td></tr><tr><td></td><td>(iii) Karala (fragment)</td><td>Dinhata</td><td>Phulbari</td><td>8</td><td></td></tr><tr><td rowspan="2">46.</td><td>(i) Sipprasad Mustati</td><td>Dinhata</td><td>Phulbari</td><td>8</td><td>373.2</td></tr><tr><td>(ii) Sipprasad Mustati (Fragment)</td><td>Dinhata</td><td>Phulbari</td><td>6</td><td></td></tr><tr><td rowspan="8">47.</td><td>(i) Dakshin Masaldanga</td><td>Dinhata</td><td>Bhurungamari</td><td>6</td><td>571.38</td></tr><tr><td>(ii) Dakshin Masaldanga</td><td>Dinhata</td><td>Bhurungamari</td><td>6</td><td></td></tr><tr><td>(Fragment) (iii) Dakshin Masaldanga</td><td>Dinhata</td><td>Bhurungamari</td><td>6</td><td></td></tr><tr><td>(Fragment) (iv) Dakshin Masaldanga</td><td>Dinhata</td><td>Bhurungamari</td><td>6</td><td></td></tr><tr><td>(Fragment) (v) Dakshin Masaldanga</td><td>Dinhata</td><td>Bhurungamari</td><td>6</td><td></td></tr><tr><td>(Fragment) (vi) Dakshin</td><td>Dinhata</td><td>Bhurungamari</td><td>6</td><td></td></tr><tr><td>Masaldanga (Fragment)</td><td></td><td></td><td></td><td></td></tr></table>

# (Appendix I)

# <table><tr><td>1</td><td>2</td><td>3</td><td>4</td><td>5</td><td>6</td></tr><tr><td>48.</td><td>(i) Paschim Masaldanga</td><td>Dinhata</td><td>Bhurungamari</td><td>4</td><td>29.49</td></tr><tr><td rowspan="3">49.</td><td>(ii) Paschim Masaldanga (Fragment)</td><td>Dinhata</td><td>Bhurungamari</td><td>4</td><td></td></tr><tr><td>(i) Purba Chhit Masaldanga</td><td>Dinhata</td><td>Bhurungamari</td><td>10</td><td>35.01</td></tr><tr><td>(ii) Purba Chhit Masaldanga (Fragment)</td><td>Dinhata</td><td>Bhurungamari</td><td>10</td><td></td></tr><tr><td rowspan="2">50.</td><td>(i) Purba Masaldanga</td><td>Dinhata</td><td>Bhurungamari</td><td>11</td><td>153.89</td></tr><tr><td>(ii) Purba Masaldanga (Fragment)</td><td>Dinhata</td><td>Bhurungamari</td><td>11</td><td></td></tr><tr><td rowspan="3">51.</td><td>(i) Uttar Dhaldanga</td><td>Tufanganj</td><td>Bhurungamari</td><td>14</td><td>24.98</td></tr><tr><td>(ii) Uttar Dhaldanga (Fragment)</td><td>Tufanganj</td><td>Bhurungamari</td><td>14</td><td></td></tr><tr><td>(iii) Uttar Dhaldanga (Fragment)</td><td>Tufanganj</td><td>Bhurungamari</td><td>14</td><td></td></tr><tr><td colspan="2">Total Area</td><td></td><td></td><td></td><td>7,110.02</td></tr></table>

# The above given details of enclaves have been jointly compared and reconciled with records held by India and Bangladesh during the Indo-Bangladesh Conference held at Calcutta during 9th—12th October, 1996 as well as during joint field inspection at Jalpaiguri (West Bengal) – Panchagarh (Bangladesh) sector during 21—24 November, 1996.

# Brig. J.R. Peter

# Director Land Records & Survey

# Md. Shafi Uddin

# Director General, Land Records

# (Ex officio) West Bengal, India &

# Director, Eastern Circle Survey of

# and Surveys, Bangladesh.

# India, Calcutta.
# """
# print_token_count(demo_text)
# chunks = semantic_chunker(demo_text, sim_threshold=0.4, max_chars=1600, min_chars=800)
# for i, chunk in enumerate(chunks, 1):
#     print(f"\n--- Chunk {i} [{chunk['type']}] ---")
#     print(chunk['text'])

In [28]:


# Example usage
sample_text = "This is a sample sentence for token counting."
print_token_count(sample_text)

Token count: 9


9

In [54]:
import json

with open("parentchunks2.json", "r", encoding="utf-8") as f:
    data = json.load(f)

def encode_text(text):
    # Flatten the numpy array and convert to a standard Python list for JSON compatibility
    emb = embedder.encode(text)
    return emb.tolist()

children_chunks = []

# Changed the outer loop variable to 'parent' for clarity
for parent in data:
    c=count_tokens(parent["text"]) 
    print(c)
    if c <= 400:
        children_chunks.append({
            "parent_id": parent["parent_id"],
            "child_id": f"{parent['parent_id']}_child_1", # Added child_id
            "doc_id": parent.get("doc_id", ""),
            "heading_path": parent.get("heading_path", []), 
            "text": parent["text"],
            "type": "mixed", # Default type since it wasn't processed by the chunker
            "embedding": encode_text(parent["text"])
        })
    else:
        # Pass the text to the chunker directly
        chunks = semantic_chunker(parent["text"])
        
        # Changed the inner loop variable to 'index' to avoid overwriting 'parent'
        for index, chunk in enumerate(chunks, 1):
            children_chunks.append({
                "parent_id": parent["parent_id"],
                "child_id": f"{parent['parent_id']}_child_{index}", # Unique ID for each child
                "doc_id": parent.get("doc_id", ""),
                "heading_path": parent.get("heading_path", []), 
                "text": chunk["text"],
                "type": chunk.get("type", "mixed"), # Captures if it's a table, list, etc.
                "embedding": encode_text(chunk["text"])
            })

13
34
173
95
104
95
24
54
54
29
62
48
58
93
65
239
306
46
101
143
56
29
51
42
51
27
292
334
4
213
81
192
38
229
277
157
97
140
85
77
126
47
290
311
87
176
137
142
102
120
20
295
117
35
54
52
54
132
90
51
23
524
70
37
20
123
166
50
131
298
361
395
408
306
131
439
475
316
536
445
208
325
138
260
254
133
370
429
471
408
371
391
435
582
295
14
821
2444
792
83
178
169
144
1425
413
529
2735
393
306
1364
495
472
492
138
646
271
2385
2482
575
367
311
71
979
768
158
175
2243
570
1423
103
366
273
696
429
1702
1251
263
1042
316
1600
405
815
1652
126
213
712
926
415
784
467
3391
3055
335
263
413
410
399
1292
632
991
1081
1071
110
162
428
135
216
647
616
898
476
890
837
61
436
415
492
554
58
347
100
506
902
685
502
466
435
294
153
144
349
470
1993
1117
375
779
967
1300
1941
924
477
861
284
669
171
135
588
156
216
334
505
823
420
351
493
1481
404
1820
2341
302
215
453
989
1328
624
456
211
637
374
890
396
444
354
483
534
484
1235
770
2473
370
277
282
319
330
9
746
507
129
223
152
188
103
158
9
129
10

In [61]:
len(children_chunks)
# children_chunks[0]

871

In [60]:
with open("childrenchunks.json","w") as f:
    json.dump(children_chunks,f)